In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


In [ ]:
import numpy as np


def calculate_msis_250(
    Y,
    lower,
    upper,
    context_scales,
    alpha=0.2,
):
    """
    Rolling 250-context Mean Scaled Interval Score.

    Parameters
    ----------
    Y : array, shape (n_origins, 20)
        Actual future returns.

    lower : array, shape (n_origins, 20)
        Predicted lower quantile (q0.1 for 80% intervals).

    upper : array, shape (n_origins, 20)
        Predicted upper quantile (q0.9 for 80% intervals).

    context_scales : array, shape (n_origins,)
        Mean absolute one-step naive differences
        computed from each origin's 250-point context.

    alpha : float
        Miscoverage probability.
        alpha=0.2 corresponds to 80% nominal coverage.

    Returns
    -------
    Dictionary containing:
        Overall MSIS
        Horizon-specific MSIS
        Mean interval score (unscaled)
        Empirical interval coverage
        Mean interval width
    """

    Y = np.asarray(Y, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)

    context_scales = np.asarray(
        context_scales,
        dtype=float
    )

    if not (
        Y.shape == lower.shape == upper.shape
    ):
        raise ValueError(
            "Y, lower and upper must have matching shapes."
        )

    if context_scales.shape != (Y.shape[0],):
        raise ValueError(
            "Context scale shape mismatch."
        )

    if not (0 < alpha < 1):
        raise ValueError(
            "alpha must be between 0 and 1."
        )

    if not np.all(np.isfinite(Y)):
        raise ValueError("Non-finite targets.")

    if not np.all(np.isfinite(lower)):
        raise ValueError("Non-finite lower bounds.")

    if not np.all(np.isfinite(upper)):
        raise ValueError("Non-finite upper bounds.")

    if np.any(lower > upper):
        raise ValueError(
            "Some prediction intervals have lower > upper."
        )

    if (
        not np.all(np.isfinite(context_scales))
        or np.any(context_scales <= 0)
    ):
        raise ValueError(
            "Invalid context scaling factors."
        )

    # --------------------------------------------------
    # INTERVAL WIDTH
    # --------------------------------------------------

    width = upper - lower

    # --------------------------------------------------
    # LOWER AND UPPER MIS-COVERAGE PENALTIES
    # --------------------------------------------------

    lower_penalty = (
        (2.0 / alpha)
        * np.maximum(lower - Y, 0.0)
    )

    upper_penalty = (
        (2.0 / alpha)
        * np.maximum(Y - upper, 0.0)
    )

    # --------------------------------------------------
    # UNSCALED INTERVAL SCORE
    # --------------------------------------------------

    interval_scores = (
        width
        + lower_penalty
        + upper_penalty
    )

    # --------------------------------------------------
    # ROLLING-CONTEXT SCALED INTERVAL SCORE
    # --------------------------------------------------

    scaled_scores = (
        interval_scores
        / context_scales[:, None]
    )

    # --------------------------------------------------
    # HORIZON-SPECIFIC AGGREGATION
    # --------------------------------------------------

    msis_by_horizon = np.mean(
        scaled_scores,
        axis=0
    )

    # --------------------------------------------------
    # OVERALL MSIS
    # --------------------------------------------------

    msis = float(
        np.mean(msis_by_horizon)
    )

    # --------------------------------------------------
    # ADDITIONAL DIAGNOSTICS
    # --------------------------------------------------

    inside = (
        (Y >= lower)
        & (Y <= upper)
    )

    coverage = float(
        np.mean(inside)
    )

    mean_width = float(
        np.mean(width)
    )

    mean_interval_score = float(
        np.mean(interval_scores)
    )

    return {
        "msis_250": msis,
        "msis_250_by_horizon": msis_by_horizon,
        "mean_interval_score": mean_interval_score,
        "coverage": coverage,
        "mean_width": mean_width,
    }

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

LOOKBACK = 250
HORIZON = 20
ALPHA = 0.20

PROJECT_DIR = Path(
    r"."
    r"\BForTFinProbability"
)

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

MODEL_DIRECTORIES = {
    "GARCH(1,1)": "garch_standardized_train_test",
    "LightGBM": "lightgbm_standardized_train_test",
    "Sundial": "sundial_standardized_train_test",
    "Lag-Llama": "lag-llama_standardized_train_test",
    "Chronos-T5": "chronos_standardized_train_test",
    "Toto": "toto_standardized_train_test",
    "TimesFM": "timesfm_standardized_train_test",
    "Moirai": "moirai_standardized_train_test",
    "Moirai-MoE": "moirai_moe_standardized_train_test",
    "Student-t GARCH": "garch_t_1_1_standardized_train_test",
}

MODEL_SUFFIXES = {
    "GARCH(1,1)": "_garch_1_1",
    "LightGBM": "_lightgbm",
    "Sundial": "_sundial",
    "Lag-Llama": "_lag-llama",
    "Chronos-T5": "_chronos-t5-small",
    "Toto": "_toto",
    "TimesFM": "_timesfm-2.5-200m",
    "Moirai": "_moirai-2.0-r-small",
    "Moirai-MoE": "_moirai-moe-1.0-r-base",
    "Student-t GARCH": "_garch-t-1-1-simulated",
}


# ============================================================
# 1. LOAD ORIGINAL TEST DATA
# ============================================================

def msis_load_test_xy(test_path):

    df = pd.read_csv(test_path)

    x_cols = [
        f"X_{i}" for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}" for i in range(HORIZON)
    ]

    X = df[x_cols].to_numpy(dtype=float)
    Y = df[y_cols].to_numpy(dtype=float)

    if (
        X.ndim != 2
        or X.shape[1] != LOOKBACK
        or Y.shape != (len(X), HORIZON)
    ):
        raise ValueError(
            f"Unexpected X/Y shapes: {X.shape}, {Y.shape}"
        )

    if not (
        np.all(np.isfinite(X))
        and np.all(np.isfinite(Y))
    ):
        raise ValueError(
            f"Non-finite test data: {test_path}"
        )

    return X, Y


# ============================================================
# 2. FIND QUANTILE FILES
# ============================================================

def msis_clean_garch_name(dataset_name):

    return (
        dataset_name
        .lower()
        .replace(" historical data", "")
        .replace(
            "_no_preprocess_lookback-250_horizon-20_slide-1",
            ""
        )
        .replace(" ", "_")
        .replace(".", "")
    )


def msis_find_quantile_file(
    model_name,
    model_root,
    asset_class,
    dataset_name,
):

    model_asset_dir = model_root / asset_class

    if model_name == "GARCH(1,1)":

        clean_name = msis_clean_garch_name(
            dataset_name
        )

        prefix = (
            f"{asset_class.lower()}_"
            f"{clean_name}_"
            f"garch_1_1_train_test"
        )

        expected = (
            model_asset_dir
            / f"{prefix}_quantiles.npz"
        )

        if expected.exists():
            return expected

        matches = list(
            model_asset_dir.glob(
                f"*{clean_name}*"
                f"garch_1_1_train_test*"
                f"quantiles.npz"
            )
        )

    else:

        suffix = MODEL_SUFFIXES[model_name]

        matches = list(
            model_asset_dir.glob(
                f"test_{dataset_name}"
                f"{suffix}_quantiles.npz"
            )
        )

    if len(matches) != 1:
        raise FileNotFoundError(
            f"{model_name} / {dataset_name}: "
            f"expected one quantile file, "
            f"found {len(matches)}"
        )

    return matches[0]


# ============================================================
# 3. LOAD QUANTILES
# ============================================================

def msis_load_quantiles(path):

    with np.load(path) as data:

        Q = np.asarray(
            data["quantiles"],
            dtype=float
        )

        levels = np.asarray(
            data["quantile_levels"],
            dtype=float
        )

    return Q, levels


# ============================================================
# 4. CALCULATE 250-CONTEXT SCALE
# ============================================================

def msis_context_scales(X):

    scales = np.mean(
        np.abs(np.diff(X, axis=1)),
        axis=1
    )

    if (
        not np.all(np.isfinite(scales))
        or np.any(scales <= 0)
    ):
        raise ValueError(
            "Invalid 250-context scaling factor"
        )

    return scales


# ============================================================
# 5. EXTRACT 80% INTERVAL
# ============================================================

def msis_extract_interval(Y, Q, levels):

    if Q.ndim != 3:
        raise ValueError(
            f"Expected 3D quantiles, got {Q.shape}"
        )

    if Q.shape[:2] != Y.shape:
        raise ValueError(
            f"Forecast/target mismatch: "
            f"{Q.shape} vs {Y.shape}"
        )

    if Q.shape[2] != len(levels):
        raise ValueError(
            "Quantile-level dimension mismatch"
        )

    if not np.all(np.diff(levels) > 0):
        raise ValueError(
            "Quantile levels are not increasing"
        )

    if not np.all(np.isfinite(Q)):
        raise ValueError(
            "Non-finite predicted quantiles"
        )

    idx10 = np.flatnonzero(
        np.isclose(levels, 0.1)
    )

    idx90 = np.flatnonzero(
        np.isclose(levels, 0.9)
    )

    if len(idx10) != 1 or len(idx90) != 1:
        raise ValueError(
            "Missing or duplicated q0.1/q0.9"
        )

    lower = Q[:, :, idx10[0]]
    upper = Q[:, :, idx90[0]]

    if np.any(lower > upper):
        raise ValueError(
            "Crossed 80% interval bounds"
        )

    return lower, upper


# ============================================================
# 6. CALCULATE MSIS
# ============================================================

def calculate_msis_250(
    Y,
    lower,
    upper,
    scales,
    alpha=0.2,
):

    width = upper - lower

    lower_penalty = (
        (2.0 / alpha)
        * np.maximum(lower - Y, 0)
    )

    upper_penalty = (
        (2.0 / alpha)
        * np.maximum(Y - upper, 0)
    )

    interval_score = (
        width
        + lower_penalty
        + upper_penalty
    )

    scaled_score = (
        interval_score / scales[:, None]
    )

    msis_by_horizon = np.mean(
        scaled_score,
        axis=0
    )

    msis = float(
        np.mean(msis_by_horizon)
    )

    coverage = float(
        np.mean(
            (Y >= lower) & (Y <= upper)
        )
    )

    mean_width = float(
        np.mean(width)
    )

    return {
        "msis_250": msis,
        "msis_250_by_horizon": msis_by_horizon,
        "mean_interval_score": float(
            np.mean(interval_score)
        ),
        "coverage_80": coverage,
        "mean_width": mean_width,
    }


# ============================================================
# 7. RUN DAILY AND WEEKLY
# ============================================================

for frequency in ["daily", "weekly"]:

    print()
    print("=" * 75)
    print(f"{frequency.upper()} MSIS")
    print("=" * 75)

    DATA_DIR = (
        PROJECT_DIR
        / "BForTFin_log_return"
        / "arima_split_datasets"
        / f"output_{frequency}"
    )

    BASE_DIR = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
    )

    dataset_rows = []
    horizon_rows = []
    failures = []

    for model_name, model_folder in MODEL_DIRECTORIES.items():

        model_root = BASE_DIR / model_folder

        print(f"\nModel: {model_name}")

        for folder, asset_class in ASSET_ORDER.items():

            test_files = sorted(
                (DATA_DIR / folder).glob("test_*.csv")
            )

            for test_path in test_files:

                dataset_name = (
                    test_path.stem
                    .removeprefix("test_")
                )

                try:

                    X, Y = msis_load_test_xy(
                        test_path
                    )

                    quantile_path = (
                        msis_find_quantile_file(
                            model_name,
                            model_root,
                            folder,
                            dataset_name,
                        )
                    )

                    Q, levels = msis_load_quantiles(
                        quantile_path
                    )

                    lower, upper = (
                        msis_extract_interval(
                            Y, Q, levels
                        )
                    )

                    scales = msis_context_scales(X)

                    result = calculate_msis_250(
                        Y,
                        lower,
                        upper,
                        scales,
                        alpha=ALPHA,
                    )

                    dataset_rows.append({
                        "model": model_name,
                        "asset_class": asset_class,
                        "asset_group": folder,
                        "dataset": dataset_name,
                        "msis_250": result["msis_250"],
                        "mean_interval_score": result[
                            "mean_interval_score"
                        ],
                        "coverage_80": result[
                            "coverage_80"
                        ],
                        "mean_width": result[
                            "mean_width"
                        ],
                        "n_origins": len(Y),
                    })

                    for h in range(HORIZON):

                        horizon_rows.append({
                            "model": model_name,
                            "asset_class": asset_class,
                            "dataset": dataset_name,
                            "horizon": h + 1,
                            "msis_250": float(
                                result[
                                    "msis_250_by_horizon"
                                ][h]
                            ),
                        })

                except Exception as exc:

                    failures.append({
                        "model": model_name,
                        "asset_class": asset_class,
                        "dataset": dataset_name,
                        "error": str(exc),
                    })

                    print(
                        f"ERROR: {model_name} / "
                        f"{dataset_name}: {exc}"
                    )

    # ========================================================
    # 8. SAVE DATASET-LEVEL RESULTS
    # ========================================================

    dataset_df = pd.DataFrame(dataset_rows)
    horizon_df = pd.DataFrame(horizon_rows)
    failures_df = pd.DataFrame(failures)

    if dataset_df.empty:
        raise RuntimeError(
            f"No successful MSIS results for {frequency}"
        )

    dataset_path = (
        BASE_DIR
        / f"msis_250_by_dataset_{frequency}.csv"
    )

    horizon_path = (
        BASE_DIR
        / f"msis_250_by_horizon_{frequency}.csv"
    )

    dataset_df.to_csv(
        dataset_path,
        index=False
    )

    horizon_df.to_csv(
        horizon_path,
        index=False
    )

    # ========================================================
    # 9. MODEL-LEVEL SUMMARY
    # ========================================================

    dataset_df["msis_rank"] = (
        dataset_df
        .groupby(["asset_group", "dataset"])[
            "msis_250"
        ]
        .rank(
            method="average",
            ascending=True
        )
    )

    summary_df = (
        dataset_df
        .groupby("model")
        .agg(
            datasets=("dataset", "count"),
            mean_msis=("msis_250", "mean"),
            median_msis=("msis_250", "median"),
            mean_rank=("msis_rank", "mean"),
            mean_coverage=("coverage_80", "mean"),
            mean_width=("mean_width", "mean"),
        )
        .reset_index()
        .sort_values("mean_rank")
    )

    summary_path = (
        BASE_DIR
        / f"msis_250_summary_{frequency}.csv"
    )

    summary_df.to_csv(
        summary_path,
        index=False
    )

    # ========================================================
    # 10. REPORT
    # ========================================================

    expected_datasets = sum(
        len(list(
            (DATA_DIR / folder).glob("test_*.csv")
        ))
        for folder in ASSET_ORDER
    )

    expected_rows = (
        expected_datasets * len(MODEL_DIRECTORIES)
    )

    print()
    print(f"Expected rows: {expected_rows}")
    print(f"Successful rows: {len(dataset_df)}")
    print(f"Failures: {len(failures_df)}")

    print()
    print(
        summary_df.round(5).to_string(
            index=False
        )
    )

    print()
    print(f"Saved: {dataset_path}")
    print(f"Saved: {horizon_path}")
    print(f"Saved: {summary_path}")

    if not failures_df.empty:

        failure_path = (
            BASE_DIR
            / f"msis_250_failures_{frequency}.csv"
        )

        failures_df.to_csv(
            failure_path,
            index=False
        )

        print(f"Failures saved: {failure_path}")

        raise RuntimeError(
            f"{frequency}: MSIS evaluation incomplete. "
            "Review the failure report."
        )

### Further checks

- Add the rolling empirical-distribution baseline to the MSIS evaluation.
- Decompose MSIS into scaled interval width, lower-tail penalty, and upper-tail penalty for all ten models.
- Compare Moirai and Moirai-MoE to understand why the latter has better marginal calibration but worse MSIS.
- Produce an updated leaderboard including the empirical baseline.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# MSIS DECOMPOSITION + ROLLING EMPIRICAL BASELINE
#
# Reuses functions/configuration from the successful MSIS cell:
#
#   msis_load_test_xy()
#   msis_find_quantile_file()
#   msis_load_quantiles()
#   msis_extract_interval()
#   msis_context_scales()
#   MODEL_DIRECTORIES
#   ASSET_ORDER
#   PROJECT_DIR
# ============================================================

ALPHA = 0.20
HORIZON = 20


# ============================================================
# 1. DECOMPOSE MSIS
# ============================================================

def decompose_msis_250(
    Y,
    lower,
    upper,
    scales,
    alpha=0.20,
):

    Y = np.asarray(Y, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    scales = np.asarray(scales, dtype=float)

    if not (
        Y.shape == lower.shape == upper.shape
    ):
        raise ValueError(
            "Target and interval shapes do not match."
        )

    if scales.shape != (Y.shape[0],):
        raise ValueError(
            "Scaling factors do not match origins."
        )

    if not (
        np.all(np.isfinite(Y))
        and np.all(np.isfinite(lower))
        and np.all(np.isfinite(upper))
        and np.all(np.isfinite(scales))
    ):
        raise ValueError(
            "Non-finite values detected."
        )

    if np.any(scales <= 0):
        raise ValueError(
            "Invalid context scale."
        )

    if np.any(lower > upper):
        raise ValueError(
            "Crossed interval bounds."
        )

    # --------------------------------------------------------
    # UNSCALED COMPONENTS
    # --------------------------------------------------------

    width = upper - lower

    lower_penalty = (
        2.0 / alpha
    ) * np.maximum(
        lower - Y,
        0.0
    )

    upper_penalty = (
        2.0 / alpha
    ) * np.maximum(
        Y - upper,
        0.0
    )

    # --------------------------------------------------------
    # SCALE EACH ORIGIN BEFORE AVERAGING
    # --------------------------------------------------------

    scaled_width = (
        width / scales[:, None]
    )

    scaled_lower_penalty = (
        lower_penalty / scales[:, None]
    )

    scaled_upper_penalty = (
        upper_penalty / scales[:, None]
    )

    scaled_total = (
        scaled_width
        + scaled_lower_penalty
        + scaled_upper_penalty
    )

    # --------------------------------------------------------
    # AGGREGATE
    # --------------------------------------------------------

    mean_width_component = float(
        np.mean(scaled_width)
    )

    mean_lower_component = float(
        np.mean(scaled_lower_penalty)
    )

    mean_upper_component = float(
        np.mean(scaled_upper_penalty)
    )

    total_msis = float(
        np.mean(scaled_total)
    )

    # --------------------------------------------------------
    # DIAGNOSTICS
    # --------------------------------------------------------

    below = Y < lower
    above = Y > upper

    coverage = float(
        np.mean(
            (Y >= lower) & (Y <= upper)
        )
    )

    return {
        "msis_250": total_msis,

        "scaled_width": mean_width_component,

        "lower_penalty": mean_lower_component,

        "upper_penalty": mean_upper_component,

        "coverage_80": coverage,

        "below_rate": float(np.mean(below)),

        "above_rate": float(np.mean(above)),

        "mean_width": float(np.mean(width)),

        "n_origins": len(Y),
    }


# ============================================================
# 2. ROLLING EMPIRICAL INTERVAL
# ============================================================

def empirical_interval_forecast(X):

    X = np.asarray(X, dtype=float)

    if X.ndim != 2 or X.shape[1] != 250:
        raise ValueError(
            f"Unexpected context shape: {X.shape}"
        )

    # One lower/upper quantile per origin
    q10 = np.quantile(
        X,
        0.10,
        axis=1
    )

    q90 = np.quantile(
        X,
        0.90,
        axis=1
    )

    # Repeat across all 20 horizons
    lower = np.repeat(
        q10[:, None],
        HORIZON,
        axis=1
    )

    upper = np.repeat(
        q90[:, None],
        HORIZON,
        axis=1
    )

    return lower, upper


# ============================================================
# 3. PROCESS DAILY AND WEEKLY
# ============================================================

for frequency in ["daily", "weekly"]:

    print()
    print("=" * 80)
    print(f"{frequency.upper()} MSIS DECOMPOSITION")
    print("=" * 80)

    DATA_DIR = (
        PROJECT_DIR
        / "BForTFin_log_return"
        / "arima_split_datasets"
        / f"output_{frequency}"
    )

    BASE_DIR = (
        PROJECT_DIR
        / "BForTFin_results"
        / "standardised"
        / f"output_{frequency}"
    )

    rows = []
    failures = []

    for folder, asset_class in ASSET_ORDER.items():

        test_files = sorted(
            (DATA_DIR / folder).glob("test_*.csv")
        )

        for test_path in test_files:

            dataset_name = (
                test_path.stem
                .removeprefix("test_")
            )

            try:

                X, Y = msis_load_test_xy(
                    test_path
                )

                scales = msis_context_scales(X)

                # --------------------------------------------
                # EMPIRICAL DISTRIBUTION BASELINE
                # --------------------------------------------

                empirical_lower, empirical_upper = (
                    empirical_interval_forecast(X)
                )

                result = decompose_msis_250(
                    Y,
                    empirical_lower,
                    empirical_upper,
                    scales,
                )

                rows.append({
                    "frequency": frequency,
                    "model": "Empirical Distribution",
                    "asset_class": asset_class,
                    "asset_group": folder,
                    "dataset": dataset_name,
                    **result,
                })

                # --------------------------------------------
                # ALL EXISTING MODELS
                # --------------------------------------------

                for model_name, model_folder in (
                    MODEL_DIRECTORIES.items()
                ):

                    model_root = (
                        BASE_DIR / model_folder
                    )

                    quantile_path = (
                        msis_find_quantile_file(
                            model_name,
                            model_root,
                            folder,
                            dataset_name,
                        )
                    )

                    Q, levels = msis_load_quantiles(
                        quantile_path
                    )

                    lower, upper = (
                        msis_extract_interval(
                            Y,
                            Q,
                            levels,
                        )
                    )

                    result = decompose_msis_250(
                        Y,
                        lower,
                        upper,
                        scales,
                    )

                    rows.append({
                        "frequency": frequency,
                        "model": model_name,
                        "asset_class": asset_class,
                        "asset_group": folder,
                        "dataset": dataset_name,
                        **result,
                    })

            except Exception as exc:

                failures.append({
                    "asset_class": asset_class,
                    "dataset": dataset_name,
                    "error": str(exc),
                })

                print(
                    f"ERROR: {dataset_name}: {exc}"
                )

    # ========================================================
    # 4. RESULTS DATAFRAME
    # ========================================================

    df = pd.DataFrame(rows)

    if failures:
        print(pd.DataFrame(failures).to_string(index=False))
        raise RuntimeError(
            f"{frequency}: decomposition incomplete. "
            "No outputs were saved."
        )

    expected_datasets = sum(
        len(list(
            (DATA_DIR / folder).glob("test_*.csv")
        ))
        for folder in ASSET_ORDER
    )

    expected_models = (
        len(MODEL_DIRECTORIES) + 1
    )

    expected_rows = (
        expected_datasets * expected_models
    )

    if len(df) != expected_rows:
        raise RuntimeError(
            f"Expected {expected_rows} rows, got {len(df)}"
        )

    if df.duplicated(
        ["asset_group", "dataset", "model"]
    ).any():
        raise RuntimeError(
            "Duplicate dataset-model rows detected."
        )

    # ========================================================
    # 5. VERIFY DECOMPOSITION
    # ========================================================

    reconstructed = (
        df["scaled_width"]
        + df["lower_penalty"]
        + df["upper_penalty"]
    )

    if not np.allclose(
        df["msis_250"],
        reconstructed,
        rtol=1e-10,
        atol=1e-10,
    ):
        raise RuntimeError(
            "MSIS decomposition does not sum correctly."
        )

    # ========================================================
    # 6. COMPARE AGAINST PREVIOUS MSIS RESULTS
    # ========================================================

    previous_path = (
        BASE_DIR
        / f"msis_250_by_dataset_{frequency}.csv"
    )

    if previous_path.exists():

        previous = pd.read_csv(previous_path)

        check = df.merge(
            previous[
                [
                    "model",
                    "asset_group",
                    "dataset",
                    "msis_250",
                ]
            ],
            on=[
                "model",
                "asset_group",
                "dataset",
            ],
            how="inner",
            suffixes=("_new", "_old"),
            validate="one_to_one",
        )

        if len(check) != len(previous):
            raise RuntimeError(
                "Previous MSIS results do not align "
                "with the new decomposition."
            )

        if not np.allclose(
            check["msis_250_new"],
            check["msis_250_old"],
            rtol=1e-8,
            atol=1e-8,
        ):
            raise RuntimeError(
                "New MSIS differs from previous results. "
                "Check forecast loading and alignment."
            )

        print(
            "Previous MSIS results reproduced successfully."
        )

    # ========================================================
    # 7. RANK MODELS WITHIN DATASETS
    # ========================================================

    df["rank"] = (
        df.groupby(
            ["asset_group", "dataset"]
        )["msis_250"]
        .rank(
            method="average",
            ascending=True
        )
    )

    # ========================================================
    # 8. MODEL SUMMARY
    # ========================================================

    summary = (
        df.groupby("model")
        .agg(
            datasets=("dataset", "count"),

            mean_msis=("msis_250", "mean"),

            median_msis=("msis_250", "median"),

            mean_rank=("rank", "mean"),

            scaled_width=("scaled_width", "mean"),

            lower_penalty=("lower_penalty", "mean"),

            upper_penalty=("upper_penalty", "mean"),

            mean_coverage=("coverage_80", "mean"),

            below_rate=("below_rate", "mean"),

            above_rate=("above_rate", "mean"),

            mean_width=("mean_width", "mean"),
        )
        .reset_index()
        .sort_values("mean_rank")
    )

    # ========================================================
    # 9. RELATIVE IMPROVEMENT OVER EMPIRICAL
    # ========================================================

    empirical = (
        df[
            df["model"] == "Empirical Distribution"
        ][
            [
                "asset_group",
                "dataset",
                "msis_250",
            ]
        ]
        .rename(
            columns={
                "msis_250": "empirical_msis"
            }
        )
    )

    df = df.merge(
        empirical,
        on=["asset_group", "dataset"],
        how="left",
        validate="many_to_one",
    )

    df["improvement_over_empirical_pct"] = (
        100.0
        * (
            1.0
            - df["msis_250"]
            / df["empirical_msis"]
        )
    )

    improvement_summary = (
        df.groupby("model")
        .agg(
            mean_improvement_pct=(
                "improvement_over_empirical_pct",
                "mean",
            ),

            median_improvement_pct=(
                "improvement_over_empirical_pct",
                "median",
            ),

            beats_empirical_pct=(
                "improvement_over_empirical_pct",
                lambda x: 100.0 * (x > 0).mean(),
            ),
        )
        .reset_index()
    )

    summary = summary.merge(
        improvement_summary,
        on="model",
        validate="one_to_one",
    )

    summary = summary.sort_values(
        "mean_rank"
    )

    # ========================================================
    # 10. SAVE RESULTS
    # ========================================================

    dataset_path = (
        BASE_DIR
        / f"msis_250_decomposition_{frequency}.csv"
    )

    summary_path = (
        BASE_DIR
        / f"msis_250_with_empirical_{frequency}.csv"
    )

    df.to_csv(
        dataset_path,
        index=False
    )

    summary.to_csv(
        summary_path,
        index=False
    )

    # ========================================================
    # 11. PRINT RESULTS
    # ========================================================

    print()
    print(f"Datasets: {expected_datasets}")
    print(f"Methods: {expected_models}")
    print(f"Dataset-method rows: {len(df)}")

    display_cols = [
        "model",
        "mean_msis",
        "mean_rank",
        "scaled_width",
        "lower_penalty",
        "upper_penalty",
        "mean_coverage",
        "beats_empirical_pct",
    ]

    print()
    print(
        summary[display_cols]
        .round(5)
        .to_string(index=False)
    )

    print()
    print(f"Saved: {dataset_path}")
    print(f"Saved: {summary_path}")

    # ========================================================
    # 12. MOIRAI VERSUS MOIRAI-MOE
    # ========================================================

    print()
    print("MOIRAI VS MOIRAI-MOE")

    comparison = summary[
        summary["model"].isin(
            ["Moirai", "Moirai-MoE"]
        )
    ][
        [
            "model",
            "mean_msis",
            "scaled_width",
            "lower_penalty",
            "upper_penalty",
            "mean_coverage",
            "below_rate",
            "above_rate",
        ]
    ]

    print(
        comparison
        .round(6)
        .to_string(index=False)
    )